# Working Analysis For All of the Single Phase HVDC Projects

## Imports and Setup

In [1]:
from pathlib import Path
from itertools import product

import pandas as pd
import matplotlib.pyplot as plt

from whale import Project
from whale.utilities import load_yaml

pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

%load_ext autoreload
%autoreload 2

### Project List and General Definitions

In [2]:
project_name_list = [
    # "Sunrise Wind",
    "Mayflower Wind 2",
]
metrics_list = []
inputs_list = []
library_path = Path("library").resolve()

In [3]:
metrics_configuration = {
    "# Turbines": {"metric": "n_turbines", "kwargs": {}},
    "Turbine Rating (MW)": {"metric": "turbine_rating", "kwargs": {}},
    "Project Capacity (MW)": {"metric": "capacity", "kwargs": {"units": "mw"}},
    "# OSS": {"metric": "n_substations", "kwargs": {}},
    "Total Export Cable Length (km)": {"metric": "export_system_total_cable_length", "kwargs": {}},
    "Total Array Cable Length (km)": {"metric": "array_system_total_cable_length", "kwargs": {}},
    "CapEx ($)": {"metric": "capex", "kwargs": {}},
    "CapEx per kW ($/kW)": {"metric": "capex", "kwargs": {"per_capacity": "kw"}},
    "OpEx ($)": {"metric": "opex", "kwargs": {}},
    "OpEx per kW ($/kW)": {"metric": "opex", "kwargs": {"per_capacity": "kw"}},
    "AEP (MWh)": {
        "metric": "energy_production",
        "kwargs": {"units": "mw", "aep": True, "with_losses": True}
    },
    "AEP per kW (MWh/kW)": {
        "metric": "energy_production",
        "kwargs": {"units": "mw", "per_capacity": "kw", "aep": True, "with_losses": True}
    },
    "Net Capacity Factor Without Unmodeled Losses (%)": {
        "metric": "capacity_factor",
        "kwargs": {"which": "net"}
    },
    "Net Capacity Factor With All Losses (%)": {
        "metric": "capacity_factor",
        "kwargs": {"which": "net", "with_losses": True}
    },
    "Gross Capacity Factor (%)": {"metric": "capacity_factor", "kwargs": {"which": "gross"}},
    "Energy Availability (%)": {"metric": "availability", "kwargs": {"which": "energy"}},
    "LCOE ($/MWh)": {"metric": "lcoe", "kwargs": {}},
    "IRR (%)": {"metric": "irr", "kwargs": {}},
    "NPV ($)": {"metric": "npv", "kwargs": {}},
    "Annual Losses (MWh)": {
        "metric": "energy_losses",
        "kwargs": {"with_losses": True, "units": "mw", "aep": True}
    },
}

metrics_order = [
    "# Turbines",
    "Turbine Rating (MW)",
    "Project Capacity (MW)",
    "# OSS",
    "Total Export Cable Length (km)",
    "Total Array Cable Length (km)",
    "FCR (%)",
    "Offtake Price ($/MWh)",
    "CapEx ($)",
    "CapEx per kW ($/kW)",
    "System CapEx for Export Cables ($)",
    "Installation CapEx for Export Cables ($)",
    "CapEx Without Export Cables ($)",
    "OpEx ($)",
    "OpEx per kW ($/kW)",
    "Annual OpEx per kW ($/kW)",
    "Energy Availability (%)",
    "Gross Capacity Factor (%)",
    "Net Capacity Factor Without Unmodeled Losses (%)",
    "Annual Losses (MWh)",
    "AEP (MWh)",
    "AEP per kW (MWh/kW)",
    "Net Capacity Factor With All Losses (%)",
    "LCOE ($/MWh)",
    "IRR (%)",
    "NPV ($)",
]
final_cols = ["CapEx ($)", "OpEx ($)", "Energy Production (GWh)", "Revenue ($)", "Cash Flow ($)"]

## Run Each Project and Gather the Results

In [4]:
for project_name in project_name_list:
    print(project_name)
    
    # Load the configurations
    config = load_yaml(
        library_path / "project/config",
        f"{project_name.replace(' ', '_')}_base.yaml"
    )
    config["weather_profile"] = library_path / "weather" / config["weather_profile"]
    project = Project(
        # Basic Model Configurations
        library_path=library_path,
        connect_floris_to_layout=True,
        connect_orbit_array_design=True,
        **config,
    )
    
    # Plot the layout and save the figures
    # fig, ax = project.plot_farm(return_fig=True)
    # fig.savefig(library_path / "results" / f"{project_name.lower().replace(' ', '_')}_layout.svg")
    
    # Run the project and clean up the logging
    project.run(
        which_floris="wind_rose",
        full_wind_rose=False,
        floris_reinitialize_kwargs=dict(cut_in_wind_speed=3.0, cut_out_wind_speed=25.0)
    )
    project.wombat.env.cleanup_log_files()  # Delete logging data from WOMBAT
    
    # Gather the high-level results
    report_df = project.generate_report(metrics_configuration, project_name).T
    export_system = project.orbit.system_costs["ExportCableInstallation"]
    export_installation = project.orbit.installation_costs["ExportCableInstallation"]
    capex_sans_export = project.orbit.total_capex - export_system - export_installation
    additional_reporting = pd.DataFrame(
        [
            ["FCR (%)", project.fixed_charge_rate],
            ["Offtake Price ($/MWh)", project.offtake_price],
            ["System CapEx for Export Cables ($)", export_system],
            ["Installation CapEx for Export Cables ($)", export_installation],
            ["CapEx Without Export Cables ($)", capex_sans_export],
            ["Annual OpEx per kW ($/kW)", report_df.loc["OpEx per kW ($/kW)", project_name] / project.operations_years],
        ],
        columns=["Project"] + report_df.columns.tolist(),
    ).set_index("Project")
    report_df = pd.concat((report_df, additional_reporting), axis=0).loc[metrics_order]
    
    # Gather the detailed results
    monthly_results = project.cash_flow(breakdown=True).join(project.energy_production(frequency="month-year")).fillna(0)
    monthly_results = monthly_results.assign(
        CapEx_Installation=monthly_results[[c for c in monthly_results if c.startswith("CapEx") and c.endswith("Installation")]].sum(axis=1),
        CapEx_System=monthly_results[[c for c in monthly_results if c.startswith("CapEx") and c.endswith("System")]].sum(axis=1),
    )
    monthly_results.to_csv(library_path / "results" / f"{project_name.lower().replace(' ', '_')}_monthly_detailed_results.csv")
    
    monthly_results["CapEx ($)"] = monthly_results[["CapEx_Installation", "CapEx_Soft", "CapEx_Project", "CapEx_System", "CapEx_Turbine"]].sum(axis=1)
    monthly_results = monthly_results.rename(columns={"OpEx": "OpEx ($)","Revenue": "Revenue ($)", "cash_flow": "Cash Flow ($)"})[final_cols]
    
    # Create the inputs data
    inputs = pd.DataFrame(
        [
            ["FCR", project.fixed_charge_rate],
            ["Offtake price ($/MWh)", project.offtake_price],
            ["Lease Area Price ($)", project.orbit.config["project_parameters"]["site_auction_price"]],
            ["Discount rate (%)", project.discount_rate],
            ["# Turbines", project.n_turbines()],
            ["Turbine Rating (MW)", project.turbine_rating()],
            ["Project Capacity (MW)", project.capacity("mw")],
            ["# OSS", project.n_substations()],
            ["Substructure type", "??"],
            ["Row spacing (rotor diameters)", "not used for custom layouts"],
            ["Turbine spacing (rotor diameters)", "not used for custom layouts"],
            ["Depth (m)", project.orbit.config["site"]["depth"]],
            [
                "Mean wind speed (m/s)",
                project.weather.loc[
                    project.orbit_start_date: project.wombat.env.end_datetime,
                    "windspeed_100m"
                ].mean()
            ],
            ["Distance to landfall (km)", project.orbit.config["site"]["distance_to_landfall"]],
            ["Distance to port (km)", project.wombat.config.port_distance],
            ["Interconnection distance (km)", project.orbit._phases["ElectricalDesign"]._distance_to_interconnection],
            ["# of POIs", "??"],
            ["Export cable type", [*project.orbit._phases["ElectricalDesign"].cable_lengths_by_type]],
            ["Array cable type", [*project.orbit._phases["CustomArraySystemDesign"].cable_lengths_by_type]],
        ],
        columns=["Inputs", f"{project_name}"]
    ).set_index("Inputs")
    
    # Save the outputs
    report_df.index.name = "Metrics"
    report_df.to_csv(library_path / "results" / f"{project_name.lower().replace(' ', '_')}_metrics.csv")
    inputs.to_csv(library_path / "results" / f"{project_name.lower().replace(' ', '_')}_inputs.csv")
    monthly_results.to_csv(library_path / "results" / f"{project_name.lower().replace(' ', '_')}_monthly_results.csv")
    
    # Append the high-level metrics to create a combined DataFrame later
    metrics_list.append(report_df)
    inputs_list.append(inputs)

Mayflower Wind 2
ORBIT library intialized at '/Users/rhammond/Documents/GitHub/WHaLE/examples/library'


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: /Users/rhammond/Documents/GitHub/develop_installs/ORBIT/ORBIT/phases/design/array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.FutureWarning: /Users/rhammond/Documents/GitHub/develop_installs/WOMBAT/wombat/core/post_processor.py:1327
The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.FutureWarning: /Users/rhammond/Documents/GitHub/develop_installs/WOMBAT/wombat/core/post_processor.py:1327
The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.

## Combine the High-Level Results

In [5]:
metrics_df = pd.concat(metrics_list, axis=1)
inputs_df = pd.concat(inputs_list, axis=1)

metrics_df.to_csv(library_path / "results" / "outputs_single_phase_hvdc.csv")
inputs_df.to_csv(library_path / "results" / "inputs_single_phase_hvdc.csv")

In [6]:
metrics_df

,Mayflower Wind 2
Metrics,
# Turbines,67.0000
Turbine Rating (MW),15.0000
Project Capacity (MW),"1,005.0000"
# OSS,1.0000
Total Export Cable Length (km),312.1012
Total Array Cable Length (km),159.8160
FCR (%),0.0582
Offtake Price ($/MWh),77.0000
CapEx ($),"3,498,635,149.3500"
